In [ ]:
import sys
import os

# Get the parent directory
parent_dir = os.path.abspath(os.path.join(os.getcwd(), os.pardir))

# Add parent directory to sys.path
sys.path.append(parent_dir)

print(parent_dir)


In [ ]:
import torch
import torch.optim as optim
import pandas as pd
from itables import init_notebook_mode, show
import numpy as np
import matplotlib.pyplot as plt
from sklearn.metrics import accuracy_score, precision_score, recall_score, f1_score
import importlib

import ray
from ray import tune

import deeparguing.gradual_aacbr as gradual_aacbr
import deeparguing.semantics.relu_semantics as rs
import deeparguing.base_scores.feature_weighted_base_score as fwbs
import deeparguing.casebase_edge_weights.feature_weighted_partial_order as fwpo
import deeparguing.casebase_edge_weights.learned_partial_order as lpo
import deeparguing.base_scores.learned_base_score as lbs
import deeparguing.irrelevance_edge_weights.regular_irrelevance as ri
import deeparguing.feature_extractor.simple_cnn as simple_cnn

from deeparguing.train import static_train_model, evaluate_model
from deeparguing.tune import tune_model, objective

from helper import split_data, load_mnist, cluster_data

init_notebook_mode(all_interactive=True)

In [ ]:
def reload_imports():
    importlib.reload(gradual_aacbr)
    importlib.reload(rs)
    importlib.reload(fwbs)
    importlib.reload(fwpo)
    importlib.reload(ri)
    importlib.reload(lpo)
    importlib.reload(lbs)
    importlib.reload(simple_cnn)

reload_imports()

In [ ]:
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print(device)

## Data Set

In [ ]:
SEED = 42

In [ ]:
# MNIST
X, y = load_mnist(device)
print(X.shape)
print(np.unique(y.cpu().numpy()))

In [ ]:
def visualise(sample):
    sample = np.reshape(sample.detach().cpu().numpy(), (28, 28))
    plt.imshow(sample)
    plt.show()

In [ ]:

visualise(X[1])


In [ ]:
show(y.cpu().numpy())
all_y = np.unique(y.cpu().numpy(), axis=0)

show(all_y)


## Train Model

### Split into Training, Validation and Test

In [ ]:
train_full, train, val, test = split_data(X, y, SEED)

print(f"Test Size:  {len(test['X'])}")
print(f"Train Size:  {len(train['X'])}")
print(f"Validation Size:  {len(val['X'])}")

### Cluster dataset

In [ ]:
X_centroids, y_centroids = cluster_data(train["X"].cpu().numpy(), train["y"].cpu().numpy(), lambda _: 3)

In [ ]:
# X_train = X_centroids
# y_train = y_centroids
X_centroids = torch.tensor(X_centroids, dtype=torch.float32).to(device)
y_centroids = torch.tensor(y_centroids, dtype=torch.float32).to(device)


In [ ]:
visualise(X_centroids[0])
visualise(X_centroids[-1])

In [ ]:
visualise(train["X"][0])

### Build AF

In [ ]:

# Compare against the average for each column
means = train["X"].mean(axis=0)

visualise(means)

### Train Model

In [ ]:
DEFAULT_CASE = means
# DEFAULT_CASE = torch.zeros_like(means)

X_DEFAULTS = DEFAULT_CASE.tile(len(all_y), 1, 1)
Y_DEFAULTS = torch.tensor(all_y).to(device)


In [ ]:
MAX_ITERS = 20
EPOCHS = 500
USE_SYMMETRIC_ATTACKS = False
LR = 0.0003
MOMENTUM = 0.9
SHARPNESS = 0.1111 
TORCH_SEED = 695



In [ ]:
reload_imports()
torch.manual_seed(TORCH_SEED) # TRY DIFFERENT INITIAL WEIGHTS 

# semantics = ms.MLPBasedSemantics(max_iters=MAX_ITERS, epsilon=0)
semantics = rs.ReluSemantics(max_iters=MAX_ITERS, epsilon=0)
cnn = simple_cnn.SimpleCNN()
partial_order = lpo.LearnedPartialOrder(cnn, sharpness=2)
irrelevance = ri.RegularIrrelevance(partial_order)
base_score = lbs.LearnedBaseScore(cnn)

model = gradual_aacbr.GradualAACBR(semantics, 
                                base_score,
                                irrelevance,
                                partial_order).to(device)

# criterion = torch.nn.BCELoss()
criterion = torch.nn.CrossEntropyLoss()
optimizer = optim.SGD(model.parameters(), lr=LR, momentum=MOMENTUM)

In [ ]:
reload_imports()
evaluate_model(model, X_centroids, y_centroids, X_DEFAULTS, Y_DEFAULTS, val["X"], val["y"], use_blockers=False, show_confusion=True,  print_matrix=True, print_compute_graph=True, print_graph = False )
model.plot_casebase_edge_weights_parameters()
model.plot_base_score_parameters()

In [ ]:
static_train_model(model, X_centroids, y_centroids, X_DEFAULTS, Y_DEFAULTS, optimizer, criterion, EPOCHS, X_new_cases=train["X"], y_new_cases=train["y"],   use_symmetric_attacks=False, use_blockers=False, plot_loss_curve=True)

In [ ]:
reload_imports()
with torch.no_grad():
    evaluate_model(model, X_centroids, y_centroids, X_DEFAULTS, Y_DEFAULTS, val["X"], val["y"], use_blockers=False, show_confusion=True,  print_matrix=True, print_compute_graph=True, print_graph = False )

model.plot_casebase_edge_weights_parameters()
model.plot_base_score_parameters()

In [ ]:
assert(False)

### Hyperparameter Tuning

In [ ]:
import random 

seed = random.randint(0, 1000)
torch.manual_seed(seed)
print(seed)

In [ ]:
cnn = simple_cnn.SimpleCNN()

search_space = {
    "train": static_train_model,
    "use_blockers": False,
    "no_iters": tune.choice([15, 20, 25]),
    "epochs": tune.choice([250, 750, 1500, 3000]),
    "lr": tune.loguniform(1e-4, 1e-1),
    "momentum": tune.loguniform(1e-4, 1e-1),
    "sharpness": tune.loguniform(1e-1, 1e1),
    "seed": tune.randint(0, 100),
    "use_symmetric_attacks": tune.choice([True, False]),
    "semantics": tune.sample_from(lambda config: rs.ReluSemantics(max_iters=config["no_iters"], epsilon=0)),
    "partial_order": tune.sample_from(lambda config: lpo.LearnedPartialOrder(cnn, sharpness=config["sharpness"])),
    "irrelevance": tune.sample_from(lambda config: ri.RegularIrrelevance(config["partial_order"])),
    "base_score": lbs.LearnedBaseScore(cnn),
}


In [ ]:
best_results = tune_model(search_space, X_centroids, y_centroids, X_DEFAULTS, Y_DEFAULTS, train["X"], train["y"], val["X"], val["y"], metric="f1", num_samples=10, num_gpus=1, device=device, disable_tqdm=True)

In [ ]:
print(best_results)

In [ ]:


torch.manual_seed(seed)

cnn = simple_cnn.SimpleCNN()

semantics = rs.ReluSemantics(max_iters=best_results["no_iters"], epsilon=0)
partial_order = lpo.LearnedPartialOrder(cnn, sharpness=best_results["sharpness"])
irrelevance = ri.RegularIrrelevance(partial_order)
base_score = lbs.LearnedBaseScore(cnn)

params = {
    "train": static_train_model,
    "use_blockers": best_results["use_blockers"],
    "epochs": best_results["epochs"],
    "lr": best_results["lr"],
    "momentum": best_results["momentum"],
    "use_symmetric_attacks": best_results["use_symmetric_attacks"],
    "semantics": semantics,
    "partial_order": partial_order,
    "irrelevance": irrelevance,
    "base_score": base_score,
}


objective(params, 
          X_casebase=X_centroids, 
          y_casebase=y_centroids, 
          X_default=X_DEFAULTS, 
          y_default=Y_DEFAULTS, 
          X_train_new_cases=train["X"], 
          y_train_new_cases=train["y"], 
          X_eval_new_cases=val["X"], 
          y_eval_new_cases=val["y"], 
          show_confusion=True, 
          plot_loss_curve = True,
          device=device)


### Test Set

In [ ]:
reload_imports()
objective(params, X_centroids, y_centroids, X_DEFAULTS, Y_DEFAULTS, train_full["X"], train_full["y"], test["X"], test["y"], show_confusion=True, plot_loss_curve = True, device=device)

## Baselines

In [ ]:
from sklearn import neighbors
from sklearn import tree
from sklearn.linear_model import LogisticRegression

X_test = test["X"]
y_test = test["y"]

y_centroids_label = torch.argmax(y_centroids, dim=1)

logistic_regression = LogisticRegression()
logistic_regression.fit(X_centroids.cpu(), y_centroids_label.cpu())
logistic_predictions = logistic_regression.predict(X_test)
print('Accuracy with Logistic Regression is %.2f' % (accuracy_score(np.argmax(y_test, axis=1), logistic_predictions) * 100))
print('Precision with Logistic Regression is %.2f' % (precision_score(np.argmax(y_test, axis=1), logistic_predictions, average="macro") * 100))
print('Recall with Logistic Regression is %.2f' % (recall_score(np.argmax(y_test, axis=1), logistic_predictions, average="macro") * 100))
print('F1 with Logistic Regression is %.2f' % (f1_score(np.argmax(y_test, axis=1), logistic_predictions, average="macro") * 100))

# %% KNeighborsClassifier
k_classifier = neighbors.KNeighborsClassifier()
k_classifier.fit(X_centroids, y_centroids_label)
k_neighbors_predictions = k_classifier.predict(X_test)
print('Accuracy with KNeighborsClassifier is %.2f' % (accuracy_score(np.argmax(y_test, axis=1), k_neighbors_predictions) * 100))

# %% DecisionTreeClassifier
decision_classifier = tree.DecisionTreeClassifier()
decision_classifier.fit(X_centroids, y_centroids_label)
decision_classifier_predictions = decision_classifier.predict(X_test)
print('Accuracy with DecisionTreeClassifier is %.2f' % (accuracy_score(np.argmax(y_test, axis=1), decision_classifier_predictions) * 100))
